# Python

All 17 Python examples from [docs/extensions/python.md](https://platob.github.io/yggdryl/extensions/python/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
from yggdryl import DataType, Field, Url

# Every argument accepts the obvious Python spelling of itself.
schema = Field("row", DataType.from_fields([Field("id", "int64", nullable=False)]), nullable=False)
location = Url.from_path("C:/market data/trades.arrows")

assert schema.data_type.id == "struct"
assert str(location) == "file:///C:/market%20data/trades.arrows"
assert str(location.media_type.base) == "application/vnd.apache.arrow.stream"

## Inference at the boundary

In [ ]:
from yggdryl import DataType, Field, MediaType, MimeType, Url

# A datatype expression is a datatype.
assert str(Field("id", "int64", nullable=False).data_type) == "int64"
assert DataType("list<int32>").id == "list"

# A media type is its canonical name.
assert str(MimeType("application/json")) == "application/json"
assert str(MediaType("application/json")) == "application/json"

# A path is a location.
assert str(Url.from_path("C:/tmp/a.json")) == "file:///C:/tmp/a.json"

## What a Python value becomes

In [ ]:
import datetime as dt
import zoneinfo
from decimal import Decimal

from yggdryl import json

value = {
    "price": Decimal("10.50"),
    "on": dt.date(2026, 8, 15),
    "since_midnight": dt.time(12, 30),
    "took": dt.timedelta(seconds=90),
    "at": dt.datetime(2026, 8, 15, 12, 30, tzinfo=zoneinfo.ZoneInfo("Europe/Paris")),
    "payload": b"\x00\xff",
}

restored = json.loads(json.dumps(value))

# The scale is data, so a price written to two places comes back to two.
assert str(restored["price"]) == "10.50"
assert restored["payload"] == value["payload"]
# A temporal travels as its classic ISO string, the loosely typed deal a
# schemaless wire makes; a record class or a schema recovers the typed
# reading. The zone survives as the zone name, not as the offset it
# happened to be at.
assert restored["on"] == "2026-08-15"
assert restored["took"] == "PT90.000000S"
assert restored["at"] == "2026-08-15T12:30:00.000000+02:00[Europe/Paris]"

## What a Python value loses

In [ ]:
import pathlib
import uuid
from collections import deque

from yggdryl import json

value = {
    "tags": {"b", "a"},
    "queue": deque([1, 2], maxlen=8),
    "id": uuid.UUID("12345678-1234-5678-1234-567812345678"),
    "path": pathlib.PurePosixPath("lake/trades.arrow"),
}

restored = json.loads(json.dumps(value))

assert restored == {
    "tags": ["a", "b"],
    "queue": [1, 2],
    "id": "12345678-1234-5678-1234-567812345678",
    "path": "lake/trades.arrow",
}

## Reading a class back

In [ ]:
from yggdryl import json, record

@record
class Trade:
    trade_id: int
    symbol: str

encoded = Trade(1, "AAPL").into_json()

# Without a target the document is what it says it is: data.
assert json.loads(encoded) == {"trade_id": 1, "symbol": "AAPL"}
assert json.loads(encoded, cls=Trade) == Trade(1, "AAPL")
assert Trade.from_json(encoded) == Trade(1, "AAPL")

## Field metadata is a mapping

In [ ]:
from yggdryl import Field

field = Field("trade", "int64", nullable=False, metadata={"source": "book"})
field.metadata["venue"] = "XPAR"

assert field.metadata["source"] == "book"
assert "venue" in field.metadata
assert len(field.metadata) == 2
assert sorted(field.metadata.keys()) == ["source", "venue"]
assert dict(field.metadata.items())["venue"] == "XPAR"

del field.metadata["venue"]
assert "venue" not in field.metadata

In [ ]:
from yggdryl import Field

field = Field("price", "int64", nullable=False)
field.iceberg["doc"] = "closing price"
field.postgres.update({"type": "numeric"})

assert field.iceberg["doc"] == "closing price"
assert dict(field.postgres.items()) == {"type": "numeric"}
assert len(field.iceberg) == 1
assert "doc" not in field.postgres

# The bare name is all the view needs; the full key is what the field stores.
assert field.iceberg.key("doc") == "iceberg:doc"
assert field.metadata["iceberg:doc"] == "closing price"
assert len(field.metadata) == 2

del field.iceberg["doc"]
assert not field.iceberg

In [ ]:
from yggdryl import DataType, Field

schema = Field(
    "row",
    DataType.from_fields([
        Field("year", "int32", nullable=False),
        Field("price", "int64", nullable=False),
    ]),
    nullable=False,
).with_partition_fields(["year"])

assert schema.partition_field_names == ["year"]
assert schema.data_type["year"].is_partition
assert len(schema.without_partition_fields().data_type) == 1

## Records

In [ ]:
from yggdryl import record, to_dict

@record
class Trade:
    trade_id: int
    symbol: str

trade = Trade(trade_id=1, symbol="AAPL")

assert to_dict(trade) == {"trade_id": 1, "symbol": "AAPL"}
assert Trade.schema_field().name == "Trade"
assert [field.name for field in Trade.schema_fields()] == ["trade_id", "symbol"]

## Errors

In [ ]:
from yggdryl import DataType

try:
    DataType("decimal(0,0)")
except ValueError as error:
    assert "precision" in str(error)
else:
    raise AssertionError("an invalid precision must be reported")

## `pathlib`-shaped storage

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase, Url

root = pathlib.Path(tempfile.mkdtemp())

# Construction touches nothing, so a missing location is empty, not an error.
handle = IOBase(root / "trades.arrows")
assert not handle.exists()
assert handle.read_bytes() == b""

handle.write_text("AAPL")
assert handle.read_text() == "AAPL"
assert handle.size == 4

# Random access needs no mode.
handle.pwrite(0, b"MSFT")
assert handle.pread(0, 4) == b"MSFT"

# Children resolve the way they do for a Path.
lake = IOBase(root / "lake" / "year=2024")
lake.mkdir()
(lake / "part-0.arrows").touch()
assert [entry.name for entry in lake.iterdir()] == ["part-0.arrows"]
assert len(list(IOBase(root / "lake").rglob("*.arrows"))) == 1

In [ ]:
from yggdryl import Url

url = Url("file:///lake/trades/part-0.tar.gz")

assert url.name == "part-0.tar.gz"
assert url.suffix == ".gz"
assert url.suffixes == (".tar", ".gz")
assert url.parts == ("lake", "trades", "part-0.tar.gz")
assert str(url.parent) == "file:///lake/trades"
assert str(url.with_suffix(".parquet")) == "file:///lake/trades/part-0.tar.parquet"
assert url.match("*.gz")
assert url.relative_to(Url("file:///lake")) == "trades/part-0.tar.gz"

In [ ]:
import pathlib
import tempfile

import pyarrow.fs as pafs

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp())
handle = IOBase.from_arrow_fs(pafs.LocalFileSystem(), (root / "trades.arrows").as_posix())

# An Arrow filesystem replaces whole files, so the write publishes on close.
with handle:
    handle.write_bytes(b"AAPL")

assert handle.read_bytes() == b"AAPL"
assert (root / "trades.arrows").read_bytes() == b"AAPL"

# IOBase(fs, path) infers the same thing the classmethod spells out.
assert str(IOBase(pafs.LocalFileSystem(), (root / "trades.arrows").as_posix()).url) == str(handle.url)

## Records cross as PyArrow readers

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string(), nullable=False),
])
batch = pa.record_batch({"id": [1, 2], "venue": ["XNAS", "XNYS"]}, schema=schema)

root = pathlib.Path(tempfile.mkdtemp())

# The handle's name picks the encoding; no call takes a format argument.
for name in ("trades.arrows", "trades.parquet"):
    with IOBase(root / name) as handle:
        handle.write_arrow_batch_reader(batch)
        assert handle.read_arrow_batch_reader().read_all() == pa.Table.from_batches([batch])

# A schema on the options selects and casts in one pass: the columns it leaves
# out are skipped rather than read and discarded.
handle = IOBase(root / "trades.parquet")
options = handle.record_options()
options.schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])
assert handle.read_arrow_batch_reader(options=options).schema.names == ["id"]

## Anything in, a reader out

In [ ]:
import pathlib
import tempfile

import pyarrow as pa
import pyarrow.dataset as pads

from yggdryl import IOBase

schema = pa.schema([pa.field("id", pa.int64(), nullable=False), pa.field("venue", pa.string())])
table = pa.table({"id": [1, 2], "venue": ["XNAS", None]}, schema=schema)
root = pathlib.Path(tempfile.mkdtemp())

# A table, a dataset, a generator of tables, and plain rows all write.
for name, rows in (
    ("table.parquet", table),
    ("dataset.parquet", pads.dataset(table)),
    ("generated.parquet", (chunk for chunk in table.to_batches())),
    ("rows.parquet", [{"id": 1, "venue": "XNAS"}, {"id": 2, "venue": None}]),
):
    handle = IOBase(root / name)
    handle.write_arrow(rows)
    assert handle.read_arrow().read_all().num_rows == 2

## pandas and polars

In [ ]:
import pathlib
import tempfile

import pandas as pd

from yggdryl import IOBase

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.parquet")

handle.write_pandas_frame(pd.DataFrame({"id": [1, 2], "venue": ["XNAS", "XNYS"]}))

# The plural name streams: one frame per batch, converted when it is pulled.
assert sum(len(frame) for frame in handle.read_pandas()) == 2
# The `_frame` name is the whole thing in one frame.
assert list(handle.read_pandas_frame()["venue"]) == ["XNAS", "XNYS"]

## An Iceberg table end to end

In [ ]:
import pathlib
import shutil
import tempfile

import pyarrow as pa

from yggdryl.iceberg import Catalog

warehouse = pathlib.Path(tempfile.mkdtemp(prefix="yggdryl-doc-")) / "warehouse"
catalog = Catalog(warehouse)

# Rows and a dotted name are enough: the first append creates the table.
columns = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string()),
])
table = catalog.append(
    "nyc.trades", pa.table({"id": [1, 2], "venue": ["XNAS", "XNYS"]}, schema=columns)
)
past = table.current_snapshot.snapshot_id
table.append(pa.table({"id": [3], "venue": [None]}, schema=columns))
assert catalog.list_tables("nyc") == ["nyc.trades"]
assert table.scan().read_all().num_rows == 3

# A column change is recorded on the update and committed once, on exit.
with table.update_schema() as update:
    update.add_column("", "price: float64")
assert table.scan().read_all().column("price").to_pylist() == [None, None, None]

# Undersized files rewrite as one replace commit that reports itself.
compaction = table.compact()
assert (compaction.files_before, compaction.files_after) == (2, 1)
assert table.scan().read_all().num_rows == 3

# And nothing rewrote history: the first snapshot reads as it was written.
assert table.scan_at(past).read_all().column("id").to_pylist() == [1, 2]

shutil.rmtree(warehouse.parent)